# Crystal Voxel Representation (CVR) 3D images

Converting a crystal structure (CIF) into a 3D image (NumPy array) with an arbitrary box size, resolution and channels.

> Requires the dev installation (see README): `uv pip install -e ".[dev]"`

In [2]:
import warnings
# warnings.filterwarnings('ignore', category=FutureWarning)  # pandas deprecation warnings

import numpy as np
from xiespp import CVR, get_test_samples

## 1. Reading the crystal structure
Any format readable by ASE can be used (`format='vasp'` for POSCAR files, etc.).
ASE Atoms and PyMatGen Structure objects can also be passed directly to `CVR.ThreeDImage`.

In [3]:
cif_file = sorted(get_test_samples('GaN'))[1]
atoms = CVR.crystal_parser(filepath=cif_file)
atoms

Atoms(symbols='Ga4N4', pbc=True, cell=[[4.46319, 0.0, 0.0], [0.0, 2.57362, 0.0], [-2.974371878531753, 0.0, 4.200856049164204]], spacegroup_kinds=...)

## 2. Creating the image
- `box`: a cubic box with side length `box_size` (Angstrom) divided into `n_bins`^3 voxels
- `channels`: one image channel per property, in the given order. Available: `atomic_number`, `group`, `period`, `atomic_index`
- `filling`: how the box is filled with atoms
  - `fill-cut`: the unit cell is replicated to fill the whole box, atoms outside the box are removed
  - `vacuum`: a single unit cell in the box

In [4]:
image = CVR.ThreeDImage(
    atoms=atoms,
    box=CVR.BoxImage(box_size=20, n_bins=64),  # 20 A box, 64 x 64 x 64 voxels (resolution: 20 / 64 A)
    channels=['atomic_number', 'group', 'period'],
    filling='fill-cut',
)
# The resolution must be fine enough to separate the nearest atoms, and the crystal must fit in the box
image.check_requirements(return_errors=True)

[]

In [5]:
array = image.get_image()  # normalization=False for the raw channel values
print('shape:', array.shape, '| dtype:', array.dtype, '| occupied voxels:', np.count_nonzero(array[..., 0]))

shape: (64, 64, 64, 3) | dtype: float32 | occupied voxels: 1440


## 3. Saving and loading the image

In [6]:
np.save('GaN_cvr.npy', array)
np.load('GaN_cvr.npy').shape

(64, 64, 64, 3)

## 4. Visualizing the image with plotly
Each occupied voxel is drawn as a cube colored by the value of the selected channel.

In [7]:
fig = CVR.vv.voxel_interactive_plotly(array, channels=0, box_outline=True)  # channel 0: atomic number
fig.update_layout(title='GaN - channel: atomic number', width=700, height=600)
fig.show()

In [8]:
fig = CVR.vv.voxel_interactive_plotly(array, channels=1, box_outline=True)  # channel 1: group
fig.update_layout(title='GaN - channel: group', width=700, height=600)
fig.show()

2D slices through the image (use the slider to move along the axis):

In [9]:
fig = CVR.vv.image_slice_view_plotly(array[::2, ::2, ::2], channels=0)  # down-sampled to keep the notebook small
fig.show()

The atoms (point cloud) that were placed in the box:

In [10]:
fig = CVR.vv.point_cloud_viewer(image.get_point_cloud(), box=image.box, box_outline=True)
fig.show()

## 5. Randomly rotated image
The models use rotational ensembles: the crystal is randomly rotated before creating each image.

In [11]:
image.get_rotational_point_cloud()
rotated = image.get_image(random_rotation=True)
fig = CVR.vv.voxel_interactive_plotly(rotated, channels=0, box_outline=True)
fig.update_layout(title='GaN - randomly rotated', width=700, height=600)
fig.show()

## 6. Images used by the models
The same images can be created with the parameters of the trained models.

In [12]:
from xiespp import synthesizability_2, formation_energy

for name, params in [('synthesizability (v2)', synthesizability_2.IMAGE_PARAMS),
                     ('formation energy', formation_energy.DEFAULT_IMAGE_PARAMS)]:
    img = CVR.ThreeDImage(
        atoms=atoms,
        box=CVR.BoxImage(box_size=params['box_size'], n_bins=params['n_bins']),
        channels=params['channels'],
        filling=params['filling'],
    )
    print(f"{name}: box = {params['box_size']} A, shape = {img.get_image().shape}")

synthesizability (v2): box = 50 A, shape = (128, 128, 128, 3)
formation energy: box = 17 A, shape = (32, 32, 32, 3)
